# Population periodicity-discrimination observer

This notebook presents the four regime-mixture/evidence-weighting hypotheses evaluated in the audited 200-split v5 analysis. All use the ungated hybrid observer, latent-trial mixture semantics, and the same manuscript-matching mixture prior.

- **Primary:** per-frequency regime mixtures with one set of evidence-weighting parameters for `std0` and one for `std1`, each shared across frequencies.
- **Block-invariant null:** per-frequency mixtures with one set of evidence-weighting parameters per frequency, shared across blocks.
- **Crossed constrained comparator:** per-block mixtures with one set of evidence-weighting parameters per frequency, shared across blocks.
- **Per-panel upper bound:** frequency-by-block mixtures and evidence-weighting parameters.

All four are plotted initially. The primary model is emphasized because it is the lowest-complexity model and dominates the two crossed comparators in both parameter count and held-out fit. The per-panel model achieves the lowest raw error but has 86 rather than 33 parameters; it is retained to show what frequency-by-standard flexibility adds, particularly the recovery of increasing rate reliance from low to high frequencies.

The earlier comprehensive observer run is loaded separately only for the fixed-single and rate/timing ablation controls that were not repeated in v5.

In [ ]:
# Setup and validated Longleaf outputs
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
})


def find_repo_root():
    here = Path.cwd().resolve()
    relative = Path("results/discrimination/scope/validation.json")
    for candidate in (here, *here.parents):
        if (candidate / relative).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative}")


ROOT = find_repo_root()
os.chdir(ROOT)
RESULTS = ROOT / "results/discrimination/scope"
CONTROL_RESULTS = ROOT / "results/discrimination/control"

validation = json.loads((RESULTS / "validation.json").read_text(encoding="utf-8"))
if validation.get("status") != "ready" or validation.get("failed_observer_fits", 0) != 0:
    raise RuntimeError(f"Population observer validation failed: {validation}")
if validation.get("observer_fits") != 800 or validation.get("expected_mc") != 200:
    raise RuntimeError(f"Unexpected v5 run dimensions: {validation}")

rmse = pd.read_csv(RESULTS / "population_observer_rmse.csv", low_memory=False)
rmse_mc = pd.read_csv(RESULTS / "merged_results.csv", low_memory=False)
curves = pd.read_csv(RESULTS / "population_discrimination_curves.csv", low_memory=False)
curves_mc = pd.read_csv(RESULTS / "merged_curves.csv", low_memory=False)
effective_weights = pd.read_csv(
    RESULTS / "population_observer_effective_weights.csv", low_memory=False
)
rate_share = pd.read_csv(RESULTS / "population_effective_rate_share.csv", low_memory=False)

control_validation = json.loads(
    (CONTROL_RESULTS / "validation.json").read_text(encoding="utf-8")
)
if control_validation.get("status") != "ready":
    raise RuntimeError(f"Earlier comprehensive control run is not valid: {control_validation}")
control_rmse = pd.read_csv(CONTROL_RESULTS / "population_observer_rmse.csv", low_memory=False)
control_weights = pd.read_csv(
    CONTROL_RESULTS / "population_observer_weights.csv", low_memory=False
)

FREQS = [20.0, 50.0, 100.0, 200.0]
BLOCKS = ["std0", "std1"]
REGIME_ORDER = [
    "I_first", "I_volley5", "I_volley10", "I_5", "I_10",
    "I_15", "I_25", "I_50", "I_all",
]
FREQ_COLORS = dict(zip(FREQS, plt.cm.viridis(np.linspace(0.08, 0.92, len(FREQS)))))

PRIMARY_KEY = "freq_states_standard_readout"
PANEL_KEY = "panel_states_panel_readout"
MODEL_SPECS = [
    {
        "key": PRIMARY_KEY,
        "weight_scope": "per_frequency", "readout_scope": "per_block",
        "label": "Per-frequency mixtures + per-block evidence weighting (primary)",
        "short": "Per-frequency mixtures\n+per-block evidence weighting",
        "color": "#2166ac", "ls": "-", "lw": 3.4, "alpha": 1.0,
    },
    {
        "key": PANEL_KEY,
        "weight_scope": "per_panel", "readout_scope": "per_panel",
        "label": "Per-panel mixtures + per-panel evidence weighting (upper bound)",
        "short": "Per-panel mixtures\n+per-panel evidence weighting",
        "color": "#762a83", "ls": "-", "lw": 2.4, "alpha": 0.58,
    },
    {
        "key": "standard_states_freq_readout",
        "weight_scope": "per_block", "readout_scope": "per_frequency",
        "label": "Per-block mixtures + per-frequency evidence weighting",
        "short": "Per-block mixtures\n+per-frequency evidence weighting",
        "color": "#b2182b", "ls": "-", "lw": 1.9, "alpha": 0.52,
    },
    {
        "key": "freq_states_freq_readout",
        "weight_scope": "per_frequency", "readout_scope": "per_frequency",
        "label": "Per-frequency mixtures + per-frequency evidence weighting (block-invariant null)",
        "short": "Per-frequency mixtures\n+per-frequency evidence weighting",
        "color": "#7f7f7f", "ls": ":", "lw": 2.0, "alpha": 0.45,
    },
]
MODEL_BY_KEY = {spec["key"]: spec for spec in MODEL_SPECS}


def subset(frame, **conditions):
    out = frame
    for key, value in conditions.items():
        out = out[out[key].eq(value)]
    return out.copy()


def model_subset(frame, spec, **conditions):
    return subset(
        frame,
        weight_scope=spec["weight_scope"],
        readout_scope=spec["readout_scope"],
        state_mode="mixture",
        model_family="hybrid",
        rate_gate_mode="none",
        **conditions,
    )


def to_slack_choice_display(cv_values, p_correct_values, block):
    # Display the continuous Slack choice axis, including template endpoints.
    cv = np.asarray(cv_values, dtype=float)
    p = np.asarray(p_correct_values, dtype=float)
    if str(block) == "std0":
        return np.where(np.isclose(cv, 0.0), 1.0 - p, p)
    if str(block) == "std1":
        return np.where(np.isclose(cv, 1.0), 1.0 - p, p)
    raise ValueError(f"Unknown block: {block}")


def slack_choice_label(block):
    return "P(choose CV1)" if str(block) == "std0" else "P(choose CV0)"


print(
    f"Validated v5 full run: {validation['observer_fits']:,}/{validation['observer_fits']:,} "
    f"fits converged across {validation['expected_mc']} MC splits."
)
print(
    "Mixture diversity after frequency-specific response-equivalence collapse: "
    f"minimum K_eff={validation['mixture_diversity']['effective_regime_count_min']:.2f}, "
    f"maximum class weight={validation['mixture_diversity']['max_regime_weight_max']:.2f}."
)

## Competing observer hypotheses

All four prespecified mixture/evidence-weighting scopes are shown below. The bold primary curve isolates block-dependent evidence use with only two sets of evidence-weighting parameters. The crossed constrained model asks whether block-dependent regime mixtures are sufficient when evidence weighting is shared between blocks. The per-frequency/per-frequency model is retained as a block-invariant null control, and the per-panel model is an interaction upper bound.

In [ ]:
# Behavioral fits for the three competing mixture/evidence-weighting hypotheses
BEHAVIOR_MODEL_SPECS = [
    MODEL_BY_KEY["standard_states_freq_readout"],
    MODEL_BY_KEY[PRIMARY_KEY],
    MODEL_BY_KEY[PANEL_KEY],
]
model_curves = {
    spec["key"]: model_subset(curves, spec, split="test")
    for spec in BEHAVIOR_MODEL_SPECS
}
model_curves_mc = {
    spec["key"]: model_subset(curves_mc, spec, split="test")
    for spec in BEHAVIOR_MODEL_SPECS
}
primary_curves = model_curves[PRIMARY_KEY]

fig, axes = plt.subplots(2, 4, figsize=(17, 7.4), sharex=True, sharey=True)
for row, block in enumerate(BLOCKS):
    for col, freq in enumerate(FREQS):
        ax = axes[row, col]
        reference = primary_curves[
            primary_curves["block"].eq(block)
            & np.isclose(primary_curves["freq"], freq)
        ].sort_values("cv")
        cv = reference["cv"].to_numpy(float)
        observed = to_slack_choice_display(cv, reference["p_observed_mean"], block)
        ax.errorbar(
            cv, observed, yerr=reference["p_observed_sem_mean"],
            fmt="o", color="black", capsize=3, ms=5, zorder=6,
        )

        # Draw the primary last so its heavier curve remains visible.
        for spec in [spec for spec in BEHAVIOR_MODEL_SPECS if spec["key"] != PRIMARY_KEY] + [MODEL_BY_KEY[PRIMARY_KEY]]:
            panel = model_curves[spec["key"]]
            panel = panel[
                panel["block"].eq(block) & np.isclose(panel["freq"], freq)
            ].sort_values("cv")
            panel_cv = panel["cv"].to_numpy(float)
            predicted = to_slack_choice_display(
                panel_cv, panel["p_predicted_mean"], block
            )
            ax.plot(
                panel_cv, predicted,
                color=spec["color"], ls=spec["ls"], lw=spec["lw"], alpha=spec["alpha"],
                zorder=5 if spec["key"] == PRIMARY_KEY else 3,
            )
            if spec["key"] in {PRIMARY_KEY, PANEL_KEY}:
                raw_panel = model_curves_mc[spec["key"]]
                raw_panel = raw_panel[
                    raw_panel["block"].eq(block)
                    & np.isclose(raw_panel["freq"], freq)
                ].copy()
                raw_panel["display_predicted"] = to_slack_choice_display(
                    raw_panel["cv"], raw_panel["p_predicted"], block
                )
                interval = (
                    raw_panel.groupby("cv", observed=True)["display_predicted"]
                    .quantile([0.025, 0.975]).unstack().reindex(panel_cv)
                )
                ax.fill_between(
                    panel_cv,
                    np.clip(interval[0.025].to_numpy(float), 0, 1),
                    np.clip(interval[0.975].to_numpy(float), 0, 1),
                    color=spec["color"],
                    alpha=0.16 if spec["key"] == PRIMARY_KEY else 0.08,
                    linewidth=0,
                )

        ax.set_title(f"{block} | {int(freq)} Hz")
        ax.set_ylim(0.0, 1.02)
        if row == 1:
            ax.set_xlabel("CV")
        if col == 0:
            ax.set_ylabel(slack_choice_label(block))

legend_handles = [
    Line2D([0], [0], color=spec["color"], ls=spec["ls"], lw=spec["lw"])
    for spec in BEHAVIOR_MODEL_SPECS
] + [Line2D([0], [0], marker="o", color="black", lw=0)]
legend_labels = [spec["label"] for spec in BEHAVIOR_MODEL_SPECS] + ["Behavior"]
legend = fig.legend(
    legend_handles, legend_labels,
    frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.02),
    ncol=3, fontsize=9,
)
legend.get_texts()[1].set_fontweight("bold")
fig.suptitle("Population discrimination: regime-mixture and evidence-weighting hypotheses", fontsize=15)
fig.tight_layout(rect=(0, 0.09, 1, 0.97))
plt.show()

In [ ]:
# Held-out performance and complexity across all four hypotheses
overall_mask_mc = (
    rmse_mc["freq"].astype(str).eq("ALL")
    & rmse_mc["block"].astype(str).eq("ALL")
    & rmse_mc["split"].eq("test")
)

performance_rows = []
performance_mc = {}
for spec in MODEL_SPECS:
    values = model_subset(rmse_mc[overall_mask_mc], spec)
    if values["mc"].nunique() != 200:
        raise RuntimeError(f"Incomplete MC coverage for {spec['label']}")
    performance_mc[spec["key"]] = values.set_index("mc")["rmse"].sort_index()
    performance_rows.append({
        "Model": spec["label"],
        "Model key": spec["key"],
        "Parameters": int(values["n_parameters"].iloc[0]),
        "Mixture parameters": int(values["n_weight_parameters"].iloc[0]),
        "Evidence-weighting parameters": int(values["n_readout_parameters"].iloc[0]),
        "Held-out RMSE": float(values["rmse"].mean()),
        "RMSE SEM": float(values["rmse"].sem()),
        "NLL / trial": float(values["nll_per_trial"].mean()),
    })

comparison_table = pd.DataFrame(performance_rows)
primary_row = comparison_table[comparison_table["Model key"].eq(PRIMARY_KEY)].iloc[0]
panel_row = comparison_table[comparison_table["Model key"].eq(PANEL_KEY)].iloc[0]
winner_rmse = float(primary_row["Held-out RMSE"])
competitor_rmse = float(
    comparison_table[
        comparison_table["Model key"].eq("freq_states_freq_readout")
    ]["Held-out RMSE"].iloc[0]
)
rmse_reduction = competitor_rmse - winner_rmse
relative_reduction = rmse_reduction / competitor_rmse

fig, ax = plt.subplots(figsize=(10.5, 4.8))
x = np.arange(len(MODEL_SPECS))
for index, spec in enumerate(MODEL_SPECS):
    row = comparison_table[comparison_table["Model key"].eq(spec["key"])].iloc[0]
    ax.errorbar(
        index, row["Held-out RMSE"], yerr=row["RMSE SEM"],
        fmt="o", color=spec["color"], ecolor=spec["color"], capsize=5,
        markersize=12 if spec["key"] == PRIMARY_KEY else 9,
        markeredgecolor="black" if spec["key"] == PRIMARY_KEY else spec["color"],
        markeredgewidth=1.2 if spec["key"] == PRIMARY_KEY else 0,
    )
    ax.text(
        index, row["Held-out RMSE"] + 0.006,
        f"k={int(row['Parameters'])}", ha="center", va="bottom", fontsize=9,
        fontweight="bold" if spec["key"] == PRIMARY_KEY else "normal",
    )
ax.set_xticks(x, [spec["short"] for spec in MODEL_SPECS])
ax.get_xticklabels()[0].set_fontweight("bold")
ax.set_ylabel("Held-out RMSE")
ax.set_title("Predictive fit versus observer scope")
ax.set_ylim(0.055, 0.165)
plt.tight_layout()
plt.show()

display(Markdown("### Full model comparison"))
display(
    comparison_table.drop(columns="Model key").style
    .apply(
        lambda row: [
            "font-weight: bold" if "primary" in str(row["Model"]) else ""
            for _ in row
        ],
        axis=1,
    )
    .format({
        "Held-out RMSE": "{:.4f}", "RMSE SEM": "{:.4f}",
        "NLL / trial": "{:.4f}",
    })
)

for other_key in [
    "freq_states_freq_readout", "standard_states_freq_readout", PANEL_KEY,
]:
    delta = performance_mc[other_key] - performance_mc[PRIMARY_KEY]
    direction = "higher" if float(delta.mean()) > 0 else "lower"
    print(
        f"{MODEL_BY_KEY[other_key]['label']}: RMSE was {abs(float(delta.mean())):.4f} "
        f"{direction} than primary on average; primary was lower in "
        f"{int((delta > 0).sum())}/200 MC splits."
    )
print(
    "The per-panel model is the raw-error winner but uses "
    f"{int(panel_row['Parameters']) - int(primary_row['Parameters'])} additional parameters. "
    "The primary model is the parsimonious mechanistic winner."
)

## Single-regime sufficiency control

The fixed per-frequency single-regime comparator was not repeated in the focused v5 scope run. It is loaded explicitly from the earlier validated comprehensive run. It permits the optimized named regime to differ across frequency but holds that choice fixed across CV, standards, trials, and MC splits.

In [ ]:
# Mixture versus the strongest stable single-regime control
control_overall = (
    control_rmse["freq"].astype(str).eq("ALL")
    & control_rmse["block"].astype(str).eq("ALL")
    & control_rmse["split"].eq("test")
)
single_control = control_rmse[
    control_overall
    & control_rmse["weight_scope"].eq("per_frequency")
    & control_rmse["readout_scope"].eq("per_block")
    & control_rmse["state_mode"].eq("fixed_single")
    & control_rmse["model_family"].eq("hybrid")
    & control_rmse["rate_gate_mode"].eq("none")
].iloc[0]

single_table = pd.DataFrame([
    {
        "Representation": "Per-frequency regime mixture (v5 primary)",
        "Held-out RMSE": winner_rmse,
        "RMSE SEM": float(primary_row["RMSE SEM"]),
    },
    {
        "Representation": "Fixed per-frequency single regime (control run)",
        "Held-out RMSE": float(single_control["rmse_mean"]),
        "RMSE SEM": float(single_control["rmse_sem"]),
    },
])
single_improvement = float(single_control["rmse_mean"] - winner_rmse)
single_relative = single_improvement / float(single_control["rmse_mean"])

selected = control_weights[
    control_weights["weight_scope"].eq("per_frequency")
    & control_weights["readout_scope"].eq("per_block")
    & control_weights["state_mode"].eq("fixed_single")
    & control_weights["model_family"].eq("hybrid")
    & control_weights["rate_gate_mode"].eq("none")
    & (control_weights["selected_mean"] > 0.5)
].copy()
selected_label = (
    "canonical_thresh_type" if "canonical_thresh_type" in selected else "thresh_type"
)
selected = selected[["freq", selected_label]].drop_duplicates().sort_values("freq")
selected.columns = ["Frequency (Hz)", "Selected fixed regime"]

display(
    single_table.style
    .apply(
        lambda row: [
            "font-weight: bold" if "primary" in str(row["Representation"]) else ""
            for _ in row
        ], axis=1,
    )
    .format({"Held-out RMSE": "{:.4f}", "RMSE SEM": "{:.4f}"})
)
print(
    f"The mixture reduced held-out RMSE by {single_improvement:.4f} "
    f"({single_relative:.1%}) relative to the fixed single-regime control."
)
display(Markdown("### Optimized fixed regime at each frequency"))
display(selected.reset_index(drop=True))

### Population spike-count variability within each fixed regime

The fixed-single observer does not receive a deterministic response. Each selected regime uses one fixed stimulation amplitude, but the population response varies across the 100 independently generated trains at every nonzero CV. CV0 has zero replicate variance because the pipeline intentionally uses one canonical CV0 pattern for all replicates. The full range is descriptive only; it also contains the systematic effect of changing CV.


In [ ]:
# Actual population spike counts generated by the optimized fixed regimes
feature_path = ROOT / "data/features/population_cv0_based_features.csv"
equivalence_path = ROOT / "data/features/population_regime_response_equivalence.csv"
if not feature_path.is_file() or not equivalence_path.is_file():
    raise FileNotFoundError(
        "Population feature/equivalence tables are required for the spike-count summary. "
        "Run notebook 04_FeatureEngineering.ipynb first."
    )

population_features = pd.read_csv(feature_path, low_memory=False)
regime_equivalence = pd.read_csv(equivalence_path, low_memory=False)
if regime_equivalence["regime_active"].dtype == object:
    equivalence_active = regime_equivalence["regime_active"].astype(str).str.lower().isin(["true", "1", "yes"])
else:
    equivalence_active = regime_equivalence["regime_active"].astype(bool)


def count_summary(values):
    values = pd.to_numeric(values, errors="raise")
    sd = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    return "{:.1f} +/- {:.1f} [{}-{}]".format(
        float(values.mean()), sd, int(values.min()), int(values.max())
    )


spike_rows = []
for _, fixed in selected.iterrows():
    freq = float(fixed["Frequency (Hz)"])
    canonical = str(fixed["Selected fixed regime"])
    match = regime_equivalence[
        np.isclose(regime_equivalence["freq"], freq)
        & regime_equivalence["canonical_regime"].eq(canonical)
        & equivalence_active
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one active representative for {freq:g} Hz / {canonical}; found {len(match)}"
        )
    representative = str(match.iloc[0]["representative_regime"])
    response = population_features[
        np.isclose(population_features["freq"], freq)
        & population_features["thresh_type"].eq(representative)
    ].copy()
    if response["rep_index"].nunique() != 100 or response["cv"].nunique() != 11:
        raise RuntimeError(f"Incomplete response grid for {freq:g} Hz / {canonical}")
    amplitudes = response["amp_uA"].dropna().unique()
    if len(amplitudes) != 1:
        raise RuntimeError(f"Fixed regime has {len(amplitudes)} amplitudes at {freq:g} Hz")

    cv0 = response[np.isclose(response["cv"], 0.0)]["n_spikes_1ms"]
    cv1 = response[np.isclose(response["cv"], 1.0)]["n_spikes_1ms"]
    within_cv_sd = (
        response[response["cv"] > 0]
        .groupby("cv")["n_spikes_1ms"]
        .std(ddof=1)
    )
    spike_rows.append({
        "Frequency (Hz)": int(freq),
        "Fixed regime": canonical,
        "Amplitude (uA)": float(amplitudes[0]),
        "CV0 population spikes, mean +/- SD [range]": count_summary(cv0),
        "CV1 population spikes, mean +/- SD [range]": count_summary(cv1),
        "Mean within-CV SD (CV > 0)": float(within_cv_sd.mean()),
        "Full observed range": f"{int(response['n_spikes_1ms'].min())}-{int(response['n_spikes_1ms'].max())}",
    })

population_spike_table = pd.DataFrame(spike_rows)
display(
    population_spike_table.style
    .format({"Amplitude (uA)": "{:.3f}", "Mean within-CV SD (CV > 0)": "{:.1f}"})
    .hide(axis="index")
)
print(
    "The fixed-single performance comparison already incorporates these response distributions; "
    "the table characterizes within-regime variability rather than independently proving regime switching."
)


### Population spike-count and recruitment profiles across operating regimes

The paired figures separate total population output from recruited population size. Both use the same CV0-anchored operating amplitudes, regime panels, frequency colors, and mean +/- SD summary across 100 response realizations. An axon is counted as recruited when it produces at least one spike in the complete response-analysis window.


In [ ]:
# Population spike-count and recruited-axon profiles for every nominal regime
required_columns = {
    "freq", "cv", "rep_index", "thresh_type", "n_spikes_1ms",
    "active_axon_count", "n_population_axons",
}
missing_columns = required_columns.difference(population_features.columns)
if missing_columns:
    raise RuntimeError(f"Population feature table lacks required columns: {sorted(missing_columns)}")

population_response_profiles = (
    population_features[
        population_features["freq"].isin(FREQS)
        & population_features["thresh_type"].isin(REGIME_ORDER)
    ]
    .groupby(["thresh_type", "freq", "cv"], observed=True, as_index=False)
    .agg(
        mean_population_spikes=("n_spikes_1ms", "mean"),
        sd_population_spikes=("n_spikes_1ms", "std"),
        mean_active_axons=("active_axon_count", "mean"),
        sd_active_axons=("active_axon_count", "std"),
        n_reps=("rep_index", "nunique"),
    )
)
if not population_response_profiles["n_reps"].eq(100).all():
    raise RuntimeError("Expected 100 response realizations in every frequency/CV/regime cell")

population_sizes = population_features["n_population_axons"].dropna().astype(int).unique()
if len(population_sizes) != 1:
    raise RuntimeError(f"Expected one modeled population size; found {population_sizes}")
population_size = int(population_sizes[0])


def plot_population_profile_grid(mean_column, sd_column, ylabel, title, upper_limit=None):
    fig, axes = plt.subplots(3, 3, figsize=(14.5, 10.2), sharex=True)
    for ax, regime in zip(axes.flat, REGIME_ORDER):
        for freq in FREQS:
            panel = population_response_profiles[
                population_response_profiles["thresh_type"].eq(regime)
                & np.isclose(population_response_profiles["freq"], freq)
            ].sort_values("cv")
            cv = panel["cv"].to_numpy(float)
            mean = panel[mean_column].to_numpy(float)
            sd = panel[sd_column].fillna(0).to_numpy(float)
            lower = np.maximum(mean - sd, 0)
            upper = mean + sd if upper_limit is None else np.minimum(mean + sd, upper_limit)
            color = FREQ_COLORS[freq]
            ax.plot(cv, mean, marker="o", ms=3.5, lw=1.8, color=color, label=f"{int(freq)} Hz")
            ax.fill_between(cv, lower, upper, color=color, alpha=0.11, linewidth=0)
        regime_suffix = "max" if regime == "I_all" else regime[2:]
        ax.set_title(rf"$N_{{\mathrm{{{regime_suffix}}}}}$")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(bottom=0)
        if ax in axes[-1, :]:
            ax.set_xlabel("CV")
        if ax in axes[:, 0]:
            ax.set_ylabel(ylabel)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, title="Frequency", loc="lower center", ncol=len(FREQS))
    fig.suptitle(title, y=0.995, fontsize=15)
    fig.tight_layout(rect=(0, 0.07, 1, 0.97))
    plt.show()
    return fig


population_spike_figure = plot_population_profile_grid(
    "mean_population_spikes", "sd_population_spikes",
    "Population spikes / 2 s",
    "Population spike counts at CV0-anchored operating amplitudes",
)
population_recruitment_figure = plot_population_profile_grid(
    "mean_active_axons", "sd_active_axons",
    f"Recruited axons (of {population_size})",
    "Recruited population size at CV0-anchored operating amplitudes",
    upper_limit=population_size,
)


## Effective rate reliance: primary versus per-panel upper bound

The primary observer has one set of evidence-weighting parameters for each block, shared across frequencies. These parameters determine how spike-rate, burst-rate, spike-timing, and burst-timing evidence contribute to the behavioral prediction. The per-panel upper bound instead fits both a regime mixture and evidence-weighting parameters for every frequency-by-block panel. Plotting both reveals what the additional interaction freedom buys: most notably, the panel model can suppress rate reliance at 20 Hz and increase it at higher frequencies while retaining the std0/std1 contrast.

In [ ]:
# Rate-reliance curves and frequency summary for primary and per-panel models
rate_specs = [MODEL_BY_KEY[PRIMARY_KEY], MODEL_BY_KEY[PANEL_KEY]]
share_curves_mc = {
    spec["key"]: model_subset(curves_mc, spec, split="test")
    for spec in rate_specs
}
share_curves = {}
for spec in rate_specs:
    share_curves[spec["key"]] = (
        share_curves_mc[spec["key"]]
        .groupby(["freq", "block", "cv"], observed=True)["effective_rate_share"]
        .agg(
            effective_rate_share_mean="mean",
            effective_rate_share_sem="sem",
            effective_rate_share_q025=lambda values: values.quantile(0.025),
            effective_rate_share_q975=lambda values: values.quantile(0.975),
            n_mc="count",
        ).reset_index()
    )
primary_share = share_curves[PRIMARY_KEY]

fig = plt.figure(figsize=(17, 10.2))
grid = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.15], hspace=0.34, wspace=0.18)
axes = np.asarray([[fig.add_subplot(grid[row, col]) for col in range(4)] for row in range(2)])

for row, block in enumerate(BLOCKS):
    for col, freq in enumerate(FREQS):
        ax = axes[row, col]
        for spec in rate_specs:
            panel = share_curves[spec["key"]]
            panel = panel[
                panel["block"].eq(block) & np.isclose(panel["freq"], freq)
            ].sort_values("cv")
            cv = panel["cv"].to_numpy(float)
            mean = panel["effective_rate_share_mean"].to_numpy(float)
            ax.plot(
                cv, mean, color=spec["color"], lw=spec["lw"], ls=spec["ls"], alpha=spec["alpha"],
                zorder=4 if spec["key"] == PRIMARY_KEY else 3,
            )
            ax.fill_between(
                cv,
                panel["effective_rate_share_q025"].to_numpy(float),
                panel["effective_rate_share_q975"].to_numpy(float),
                color=spec["color"],
                alpha=0.16 if spec["key"] == PRIMARY_KEY else 0.10,
                linewidth=0,
            )
        ax.set_title(f"{block} | {int(freq)} Hz")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        if row == 1:
            ax.set_xlabel("CV")
        if col == 0:
            ax.set_ylabel("Effective rate share")

summary_ax = fig.add_subplot(grid[2, :])
summary_parts = []
for spec in rate_specs:
    raw = model_subset(curves_mc, spec, split="test")
    per_mc = (
        raw.groupby(["mc", "freq", "block"], as_index=False)
        ["effective_rate_share"].mean()
    )
    grouped = (
        per_mc.groupby(["freq", "block"], as_index=False)
        .agg(
            mean=("effective_rate_share", "mean"),
            sem=("effective_rate_share", "sem"),
            q025=("effective_rate_share", lambda values: values.quantile(0.025)),
            q975=("effective_rate_share", lambda values: values.quantile(0.975)),
        )
    )
    grouped["model_key"] = spec["key"]
    summary_parts.append(grouped)
share_summary_long = pd.concat(summary_parts, ignore_index=True)

model_offsets = {PRIMARY_KEY: -1.6, PANEL_KEY: 1.6}
block_markers = {"std0": "o", "std1": "s"}
for spec in rate_specs:
    for block in BLOCKS:
        panel = share_summary_long[
            share_summary_long["model_key"].eq(spec["key"])
            & share_summary_long["block"].eq(block)
        ].sort_values("freq")
        x = panel["freq"].to_numpy(float) + model_offsets[spec["key"]]
        center = panel["mean"].to_numpy(float)
        interval_error = np.vstack([
            center - panel["q025"].to_numpy(float),
            panel["q975"].to_numpy(float) - center,
        ])
        summary_ax.errorbar(
            x, center, yerr=interval_error,
            color=spec["color"], marker=block_markers[block],
            lw=spec["lw"], ls=spec["ls"], ms=7, capsize=4,
            markeredgecolor="white", markeredgewidth=0.6,
        )

summary_ax.set_xticks(FREQS, [f"{int(freq)}" for freq in FREQS])
summary_ax.set_xlim(10, 210)
summary_ax.set_ylim(-0.03, 1.03)
summary_ax.set_xlabel("Frequency (Hz)")
summary_ax.set_ylabel("Mean effective rate share across CV")
summary_ax.set_title("Frequency summary")

model_handles = [
    Line2D([0], [0], color=spec["color"], lw=spec["lw"], ls=spec["ls"])
    for spec in rate_specs
]
model_labels = [
    "Per-frequency mixtures + per-block evidence weighting (primary)",
    "Per-panel mixtures + per-panel evidence weighting (upper bound)",
]
model_legend = summary_ax.legend(
    model_handles, model_labels, frameon=False, loc="lower center", ncol=2,
)
model_legend.get_texts()[0].set_fontweight("bold")
summary_ax.add_artist(model_legend)
summary_ax.legend(
    [
        Line2D([0], [0], marker="o", color="black", lw=0),
        Line2D([0], [0], marker="s", color="black", lw=0),
    ],
    ["std0", "std1"], frameon=False, loc="upper left", title="Standard",
)

fig.suptitle("Effective rate reliance: primary model and per-panel upper bound", fontsize=16)
plt.show()

share_table = share_summary_long.pivot_table(
    index="freq", columns=["model_key", "block"], values="mean"
).reindex(index=FREQS, columns=pd.MultiIndex.from_product([
    [PRIMARY_KEY, PANEL_KEY], BLOCKS,
]))
share_table.columns = pd.MultiIndex.from_tuples([
    ("Primary: per-frequency mixtures + per-block evidence weighting", block)
    if model == PRIMARY_KEY else
    ("Upper bound: per-panel mixtures + per-panel evidence weighting", block)
    for model, block in share_table.columns
], names=["Model", "Block"])
share_table.index = share_table.index.astype(int)
share_table.index.name = "Frequency (Hz)"
share_table.loc["All frequencies"] = share_table.mean(axis=0)
display(Markdown("### Mean effective rate share across CV"))
display(share_table.style.format("{:.3f}"))

## Frequency-specific population representation

The primary spike-count regime mixture is estimated once per frequency and shared across both blocks and all CV values. The bar chart uses a fixed nine-regime legend. When nominal regimes produce response-equivalent population activity at a given frequency, their combined mass is displayed under the retained representative regime; the redundant regime therefore has no separate slice at that frequency.


In [ ]:
# Nominal-regime display of the response-distinct primary mixture
active_weights = model_subset(
    effective_weights, MODEL_BY_KEY[PRIMARY_KEY]
)

active_equivalence = regime_equivalence.loc[
    equivalence_active,
    ["freq", "canonical_regime", "representative_regime"],
].drop_duplicates()
if active_equivalence.duplicated(["freq", "canonical_regime"]).any():
    raise RuntimeError("Frequency/canonical-regime mapping is not unique")

display_weights = active_weights.merge(
    active_equivalence,
    left_on=["freq", "canonical_thresh_type"],
    right_on=["freq", "canonical_regime"],
    how="left", validate="many_to_one",
)
if display_weights["representative_regime"].isna().any():
    missing = display_weights.loc[
        display_weights["representative_regime"].isna(),
        ["freq", "canonical_thresh_type"],
    ].drop_duplicates()
    raise RuntimeError(
        f"Missing representative-regime mappings: {missing.to_dict('records')}"
    )

mixture_weights = display_weights.pivot_table(
    index="freq", columns="representative_regime", values="weight_mean", aggfunc="sum"
).reindex(index=FREQS, columns=REGIME_ORDER).fillna(0)

regime_colors = dict(zip(REGIME_ORDER, plt.cm.viridis(np.linspace(0.03, 0.97, len(REGIME_ORDER)))))
fig, ax = plt.subplots(figsize=(9.6, 5.2))
x = np.arange(len(FREQS))
bottom = np.zeros(len(FREQS), dtype=float)
for regime in REGIME_ORDER:
    values = mixture_weights[regime].to_numpy(float)
    ax.bar(
        x, values, bottom=bottom, width=0.72,
        color=regime_colors[regime], edgecolor="white", linewidth=0.55,
        label=regime.replace("I_all", "I_max"),
    )
    bottom += values

ax.set_xticks(x, [f"{int(freq)} Hz" for freq in FREQS])
ax.set_ylim(0, 1)
ax.set_ylabel("Spike-count regime mixture weight")
ax.set_xlabel("Frequency")
ax.set_title("Primary per-frequency spike-count regime mixture")
legend_handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(
    legend_handles[::-1], legend_labels[::-1],
    frameon=False, title="Nominal spike-count regime",
    bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8,
)
fig.text(
    0.5, 0.01,
    "Response-equivalent regimes are combined under the retained nominal representative.",
    ha="center", va="bottom", fontsize=8.5,
)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()


## Manuscript reporting values

Rate-only and timing-only ablations were not repeated in the focused v5 scope run. Their values below are loaded from the earlier validated comprehensive run and are retained as text-level controls rather than additional psychometric figures.

In [ ]:
# Compact cue ablations and manuscript-facing summary
ablations = control_rmse[
    control_overall
    & control_rmse["weight_scope"].eq("per_frequency")
    & control_rmse["readout_scope"].eq("per_block")
    & control_rmse["state_mode"].eq("mixture")
    & control_rmse["rate_gate_mode"].eq("none")
    & control_rmse["model_family"].isin(["rate", "timing", "hybrid"])
][["model_family", "rmse_mean", "rmse_sem", "nll_per_trial_mean"]].copy()
ablations["Observer"] = ablations["model_family"].map({
    "rate": "Rate only", "timing": "Timing only", "hybrid": "Hybrid",
})
ablations = ablations[
    ["Observer", "rmse_mean", "rmse_sem", "nll_per_trial_mean"]
].sort_values("rmse_mean")
ablations.columns = ["Observer", "Held-out RMSE", "RMSE SEM", "NLL / trial"]
display(
    ablations.style.format({
        "Held-out RMSE": "{:.4f}", "RMSE SEM": "{:.4f}", "NLL / trial": "{:.4f}",
    })
)

primary_std = (
    share_summary_long[share_summary_long["model_key"].eq(PRIMARY_KEY)]
    .groupby("block")["mean"].mean()
)
panel_low = float(
    share_summary_long[
        share_summary_long["model_key"].eq(PANEL_KEY)
        & np.isclose(share_summary_long["freq"], 20.0)
    ]["mean"].mean()
)
panel_high = float(
    share_summary_long[
        share_summary_long["model_key"].eq(PANEL_KEY)
        & share_summary_long["freq"].isin([50.0, 100.0, 200.0])
    ]["mean"].mean()
)

display(Markdown(f"""
### Concise interpretation

- With the same per-frequency regime mixture, **per-block evidence weighting** reduced held-out RMSE from **{competitor_rmse:.4f}** to **{winner_rmse:.4f}**, a **{relative_reduction:.1%} reduction**, while using 33 rather than 43 parameters.
- It outperformed both models lacking block-specific evidence weighting in **200/200** MC splits. These overlapping MC splits are robustness resamples, not independent inferential observations.
- The per-panel upper bound reduced RMSE further to **{float(panel_row['Held-out RMSE']):.4f}**, but required **{int(panel_row['Parameters'])} parameters**. Its principal qualitative addition was frequency-dependent cue allocation: mean rate reliance increased from **{panel_low:.3f}** at 20 Hz to **{panel_high:.3f}** across 50-200 Hz.
- In the parsimonious primary model, mean effective rate share was **{primary_std['std0']:.3f}** for `std0` and **{primary_std['std1']:.3f}** for `std1`, supporting standard-dependent use of rate and timing evidence.
- The frequency-specific mixture reduced RMSE by **{single_improvement:.4f} ({single_relative:.1%})** relative to the optimized fixed per-frequency single-regime control from the comprehensive control run.

The primary mechanistic inference is therefore that the two standard conditions require different weighting of rate and timing evidence. The per-panel model shows that additional frequency-by-block flexibility can improve absolute fit and recover the earlier low-to-high-frequency rate-reliance pattern, but at substantially greater complexity.
"""))